In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
 
BASE_PATH = "/lustre/fswork/projects/rech/rbw/ucw75ke/projects/GradientDistillation"
# BASE_PATH = "/Users/alex/Developpement/Internship/GradientDistillation"
BACKBONE = "dinov2_vitb"
run_name = "dinov2_vitb_latent_predlatent_medoids_adam_lr1e-3_s101"
 
data = torch.load(
    f"{BASE_PATH}/output/latent_exp/results/aqua20/{BACKBONE}/{run_name}/data.pth",
    weights_only=False, map_location="cpu",
)
print("clés data.pth:", list(data.keys()))

In [ ]:
pred_latent = data["pred_latent"].float().cpu()      # [N, 12, r, r]
latent_prior = data["latent_prior"].float().cpu()
 
# décomposition physique finale, en espace pixel (remplace snapshots[-1]["I"/"J"/"T"/"B"])
images = data["images"].float().cpu()                # I final
syn_J = data["syn_J"].float().cpu()                  # J final (radiance de scène)
syn_T = data["syn_T"].float().cpu()                  # T final (transmission)
syn_B = data["syn_B"].float().cpu()                  # B final (backscatter)
 
sample_indices = data.get("sample_indices", None)
sample_paths = data.get("sample_paths", None)
sample_init = data.get("sample_init", None)          # images initiales, déjà en tenseur
labels = data.get("labels", None)
 
print("images (I finale):", tuple(images.shape))
print("syn_J:", tuple(syn_J.shape), " syn_T:", tuple(syn_T.shape), " syn_B:", tuple(syn_B.shape))
print("pred_latent:", tuple(pred_latent.shape))

In [ ]:
pyramid_snapshots = data.get("pyramid_snapshots", [])
pyramid_steps = data.get("pyramid_snapshot_steps", list(range(len(pyramid_snapshots))))
print(f"{len(pyramid_snapshots)} pyramid_snapshots, steps: {pyramid_steps}")
 
# on ne connaît pas a priori les clés stockées dans chaque snapshot (I/J/T/B tous
# présents, ou seulement une partie liée à la pyramide de J) -> introspection
if pyramid_snapshots and isinstance(pyramid_snapshots[0], dict):
    available_keys = [k for k in ["I", "J", "T", "B"] if k in pyramid_snapshots[0]]
    print("composantes présentes dans un pyramid_snapshot:", list(pyramid_snapshots[0].keys()))
else:
    available_keys = []
 
FINAL_TENSORS = {"I": images, "J": syn_J, "T": syn_T, "B": syn_B}
 
COMPS = {"clear (J)": slice(0, 4), "bc (B)": slice(4, 8), "ill (T)": slice(8, 12)}

In [ ]:
def show_evolution(snapshots, steps, key="I", class_indices=None, figsize_per_img=1.5,
                    append_final=None):
    """Évolution temporelle des sorties décodées. key: 'I', 'J', 'T', 'B'.
    Si append_final (tenseur [N,3,R,R]) est fourni, il est ajouté comme dernière
    ligne pour représenter l'état final (indépendant de ce que trackent les
    pyramid_snapshots)."""
    rows = [s for s in snapshots if key in s] if snapshots else []
    row_steps = [st for s, st in zip(snapshots, steps) if key in s] if snapshots else []
    if append_final is not None:
        rows = rows + [{key: append_final}]
        row_steps = row_steps + ["final"]
    if not rows:
        print(f"Rien à afficher pour la clé '{key}'.")
        return
    N = rows[0][key].shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    fig, axes = plt.subplots(len(rows), len(cols),
                             figsize=(len(cols) * figsize_per_img,
                                      len(rows) * figsize_per_img),
                             squeeze=False)
    for i, (snap, step) in enumerate(zip(rows, row_steps)):
        imgs = snap[key]
        for j, c in enumerate(cols):
            axes[i][j].imshow(imgs[c].detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy())
            axes[i][j].axis("off")
        axes[i][0].set_ylabel(f"it {step}", rotation=0, labelpad=30)
        axes[i][0].axis("on")
        axes[i][0].set_xticks([]); axes[i][0].set_yticks([])
    fig.suptitle(key)
    plt.tight_layout(); plt.show()
 
 
if pyramid_snapshots:
    for k in ["I", "J", "T", "B"]:
        show_evolution(pyramid_snapshots, pyramid_steps, key=k,
                        append_final=FINAL_TENSORS.get(k))
else:
    print("Pas de pyramid_snapshots exploitables : affichage de l'état final uniquement.")
 

In [ ]:
def show_decomposition(I, J, T, B, class_indices=None, figsize_per_img=1.6):
    """Grille I / J / T / B en espace pixel pour les classes choisies.
    Remplace l'affichage d'évolution quand seul l'état final est disponible."""
    comps = {"I": I, "J (radiance)": J, "T (transmission)": T, "B (backscatter)": B}
    N = I.shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    fig, axes = plt.subplots(len(comps), len(cols),
                             figsize=(len(cols) * figsize_per_img,
                                      len(comps) * figsize_per_img),
                             squeeze=False)
    for row, (name, tensor) in enumerate(comps.items()):
        for j, c in enumerate(cols):
            img = tensor[c].clamp(0, 1).cpu()
            if img.shape[0] == 1:
                axes[row][j].imshow(img[0], cmap="gray", vmin=0, vmax=1)
            else:
                axes[row][j].imshow(img.permute(1, 2, 0))
            axes[row][j].axis("off")
        axes[row][0].set_ylabel(name, rotation=0, labelpad=40, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    fig.suptitle("Décomposition physique finale (espace pixel)")
    plt.tight_layout(); plt.show()
 
 
show_decomposition(images, syn_J, syn_T, syn_B)
 

In [ ]:
def decomposition_stats(T, B, class_names=None):
    """T et B moyens par classe, en espace pixel — l'analogue pixel du drift
    latent, utile pour vérifier la non-identifiabilité de T (drift minimal
    attendu) et l'amplitude du backscatter par classe."""
    N = T.shape[0]
    t_mean = T.flatten(1).mean(1)
    b_mean = B.flatten(1).mean(1)
    fig, ax = plt.subplots(figsize=(10, 3.2))
    x = np.arange(N)
    ax.bar(x - 0.2, t_mean.numpy(), width=0.4, label="T moyen")
    ax.bar(x + 0.2, b_mean.numpy(), width=0.4, label="B moyen")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names if class_names is not None else x,
                       rotation=60, ha="right", fontsize=7)
    ax.legend(fontsize=8)
    ax.set_title("Transmission / backscatter moyens par classe (espace pixel)")
    plt.tight_layout(); plt.show()
 
 
decomposition_stats(syn_T, syn_B, class_names=labels)
 

In [ ]:
def show_latent_channels(latent, class_idx=0, figsize_per_img=1.4):
    """Les 12 canaux latents d'une image, groupés par composante physique."""
    z = latent[class_idx]                       # [12, r, r]
    fig, axes = plt.subplots(3, 4, figsize=(4 * figsize_per_img, 3 * figsize_per_img),
                             squeeze=False)
    for row, (name, sl) in enumerate(COMPS.items()):
        block = z[sl]
        vmax = block.abs().max()
        for c in range(4):
            axes[row][c].imshow(block[c], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            axes[row][c].axis("off")
        axes[row][0].set_ylabel(name, rotation=0, labelpad=35, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    fig.suptitle(f"pred_latent — classe {class_idx}")
    plt.tight_layout(); plt.show()
 
 
show_latent_channels(pred_latent, class_idx=0)

In [ ]:
def show_drift_maps(pred, prior, class_indices=None, figsize_per_img=1.4):
    """Carte spatiale |pred - prior| par composante latente : où le latent a bougé."""
    N = pred.shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    d = (pred - prior).abs()
    fig, axes = plt.subplots(3, len(cols),
                             figsize=(len(cols) * figsize_per_img, 3 * figsize_per_img),
                             squeeze=False)
    for row, (name, sl) in enumerate(COMPS.items()):
        m = d[:, sl].mean(1)                     # [N, r, r]
        vmax = m.max()
        for j, c in enumerate(cols):
            axes[row][j].imshow(m[c], cmap="magma", vmin=0, vmax=vmax)
            axes[row][j].axis("off")
        axes[row][0].set_ylabel(name, rotation=0, labelpad=35, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    fig.suptitle("dérive latente |pred − prior| (échelle commune par composante)")
    plt.tight_layout(); plt.show()
 
 
show_drift_maps(pred_latent, latent_prior)

In [ ]:
def drift_per_class(pred, prior, class_names=None):
    """RMSE de dérive latente par classe et par composante — l'analogue latent du 'T freeze'."""
    fig, ax = plt.subplots(figsize=(10, 3.2))
    N = pred.shape[0]
    x = np.arange(N)
    w = 0.27
    for k, (name, sl) in enumerate(COMPS.items()):
        r = torch.sqrt(((pred[:, sl] - prior[:, sl]) ** 2).mean(dim=(1, 2, 3)))
        ax.bar(x + (k - 1) * w, r.numpy(), width=w, label=name)
        print(f"{name:>10}  drift RMSE moyen = {r.mean():.4f}  "
              f"(relatif = {(r.mean() / prior[:, sl].std()):.3%})")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names if class_names is not None else x, rotation=60,
                       ha="right", fontsize=7)
    ax.set_ylabel("RMSE latente"); ax.legend(fontsize=8)
    ax.set_title("Dérive par rapport à l'initialisation SLURPP (espace latent)")
    plt.tight_layout(); plt.show()
 
 
drift_per_class(pred_latent, latent_prior, class_names=labels)

In [ ]:
def latent_distributions(pred, prior):
    """Le latent reste-t-il dans le support du VAE ? (dérive vs saturation)"""
    fig, axes = plt.subplots(1, 3, figsize=(12, 3), squeeze=False)
    for k, (name, sl) in enumerate(COMPS.items()):
        ax = axes[0][k]
        ax.hist(prior[:, sl].flatten().numpy(), bins=80, alpha=0.5,
                density=True, label="prior")
        ax.hist(pred[:, sl].flatten().numpy(), bins=80, alpha=0.5,
                density=True, label="optimisé")
        ax.set_title(name, fontsize=9); ax.legend(fontsize=7)
    fig.suptitle("distribution des valeurs latentes")
    plt.tight_layout(); plt.show()
 
 
latent_distributions(pred_latent, latent_prior)

In [ ]:
def compare_to_medoids(final_I, sample_init, sample_paths=None, class_indices=None,
                       figsize_per_img=1.6):
    """Médoïde/init (haut) vs I distillé final (bas). Utilise sample_init si
    disponible (tenseur déjà chargé) ; retombe sur sample_paths + PIL sinon."""
    N = final_I.shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    fig, axes = plt.subplots(2, len(cols),
                             figsize=(len(cols) * figsize_per_img, 2 * figsize_per_img),
                             squeeze=False)
    for j, c in enumerate(cols):
        if sample_init is not None:
            init_img = sample_init[c]
            if torch.is_tensor(init_img):
                init_img = init_img.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        elif sample_paths is not None:
            init_img = np.asarray(Image.open(sample_paths[c]).convert("RGB").resize((256, 256)))
        else:
            init_img = None
        if init_img is not None:
            axes[0][j].imshow(init_img)
        axes[1][j].imshow(final_I[c].clamp(0, 1).permute(1, 2, 0).cpu())
        axes[0][j].axis("off"); axes[1][j].axis("off")
    for row, lab in enumerate(["init", "distillé"]):
        axes[row][0].set_ylabel(lab, rotation=0, labelpad=30, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    plt.tight_layout(); plt.show()
 
 
compare_to_medoids(images, sample_init, sample_paths)

In [ ]:
def residual_vs_medoid(final_I, sample_init, sample_paths=None, class_indices=None,
                       figsize_per_img=1.6):
    """|I_final − init| en pixel : localise le signal ajouté par la distillation.
    Utilise sample_init si disponible ; sinon recharge depuis sample_paths."""
    R = final_I.shape[-1]
    N = final_I.shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    fig, axes = plt.subplots(1, len(cols),
                             figsize=(len(cols) * figsize_per_img, figsize_per_img),
                             squeeze=False)
    for j, c in enumerate(cols):
        if sample_init is not None:
            ref = sample_init[c]
            if torch.is_tensor(ref):
                if ref.shape[-1] != R:
                    ref = F.interpolate(ref.unsqueeze(0), size=(R, R),
                                        mode="bilinear", align_corners=False)[0]
                ref = ref.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        elif sample_paths is not None:
            ref = np.asarray(Image.open(sample_paths[c]).convert("RGB")
                             .resize((R, R)), dtype=np.float32) / 255.0
        else:
            continue
        res = (final_I[c].clamp(0, 1).permute(1, 2, 0).numpy() - ref)
        axes[0][j].imshow(np.abs(res).mean(-1), cmap="magma", vmin=0, vmax=0.5)
        axes[0][j].axis("off")
    fig.suptitle("|I distillé − init|")
    plt.tight_layout(); plt.show()
 
 
residual_vs_medoid(images, sample_init, sample_paths)